# 07 - Replication (seeds 42/43/44 x real/synth1) + calibration middle path
Runtime: **T4 GPU**. Run all cells top to bottom; **resume-safe** (finished detectors, predictions and per-run results are cached in Drive; re-run after any disconnect).

**Part 1 - replication.** Trains every missing fold detector for each (config, seed) in a queue (up to `MAX_TRAIN_HOURS` per session; re-run the queue cell to continue), builds out-of-fold predictions, runs leave-one-board-out flagging, and asks one question: **does the load reduction of the graph features (B) vs image-only (A) hold across seeds?** Rule per (config, alpha, calibration method): B flags fewer boxes in **every** seed, the seed-averaged 95% bootstrap CI (over the 11 boards) excludes 0, B's miss rate is not worse (mean diff <= +0.005), and in at least half the seeds the per-seed CI also excludes 0. If it does not hold, drop the claim.

**Part 2 - calibration middle path.** Adds two methods between pooled split-conformal (under-covers under board shift, ~1.18x alpha in 06) and worst-board (over-covers, ~2x load):
- `board-median`: median of the per-board thresholds of the 3 calibration boards.
- `shift-inflated`: estimates the shift inflation lambda from the calibration boards only (calibrate on 2 boards, measure miss/alpha on the 3rd, rotate, average), then calibrates pooled at alpha/lambda. Aim: miss ~ 1.0x alpha with moderate load. No test-board information is used.
It then recommends the method with the lowest load whose miss/alpha is within [0.95, 1.10] for B.

Notes: seed-42 synthetic detectors may come from your earlier 04 runs (possibly unbalanced open/short sampling); seeds 43/44 are trained here with balanced sampling. The reference-free head is off by default (it failed in 06).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
D = '/content/drive/MyDrive/pcbrisk'
sys.path.insert(0, f'{D}/code')
from pcbrisk_utils import *
seed_all(42)

sh('pip -q install ultralytics scikit-image opencv-python-headless scipy scikit-learn pandas pyyaml')
import numpy as np, pandas as pd, cv2, json, time, shutil, itertools, warnings, yaml
from scipy import ndimage as ndi
from skimage.morphology import remove_small_objects
from pathlib import Path
warnings.filterwarnings('ignore')
T0 = time.time()

Mounted at /content/drive


In [ ]:
# ---------------- CONFIG (edit here) ----------------
CONFIGS = ['real', 'synth1']   # real-only and synthetic-trained detectors (balanced open/short, ratio 1.0)
SEEDS = [42, 43, 44]
TRAIN_MISSING = True      # train missing fold detectors automatically
MAX_TRAIN_HOURS = 4.0     # stop training gracefully after this; re-run the cell to continue
EPOCHS, IMGSZ, BATCH = 60, 640, 16
COPPER = 'auto'           # must match notebook 03 ('dark' | 'light' | 'auto')
RETUNE = False            # True forces engine re-tuning even if saved params exist
GRID_VERSION = 'g2'
MIN_AREA = 200
RUN_HEAD = False        # reference-free CNN head failed in 06; set True to re-test it
CRIT = [0, 1]             # open, short
NAMES = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']
ALPHAS = [0.05, 0.10]
N_BOOT = 2000
CROP, HEAD_EPOCHS = 48, 8

In [ ]:
restore_tar(f'{D}/01_prepared/prepared_v2.tar', f'{L}/data', marker='.ready_prepared_v2')
if not os.path.exists(f'{L}/raw/DeepPCB'):
    sh(f'unzip -q {D}/00_raw/deeppcb_main.zip -d {L}/raw')
mp = f'{D}/01_prepared/split_manifest_v2.csv'
assert sha256_file(mp) == open(f'{D}/01_prepared/manifest_v2.sha256').read(), 'manifest changed'
man = pd.read_csv(mp, dtype={'image_id': str})
P = Path(f'{L}/raw/DeepPCB/PCBData'); Y = Path(f'{L}/data/yolo_all')
temp_path = {p.name.replace('_temp.jpg', ''): p for p in P.glob('group*/*/*_temp.jpg')}
board_of = dict(zip(man.image_id, man.source_board)); fold_of = dict(zip(man.image_id, man.fold))
boards = sorted(man.source_board.unique()); assert len(boards) == 11
order = [str(b) for b in np.random.default_rng(42).permutation(boards)]      # same order as notebook 02

def load_gt(stem):
    rows = []
    for l in open(Y/'labels'/f'{stem}.txt'):
        v = l.split()
        if len(v) < 5: continue
        c, cx, cy, w, h = int(v[0]), *map(float, v[1:5])
        rows.append([c, (cx-w/2)*640, (cy-h/2)*640, (cx+w/2)*640, (cy+h/2)*640])
    return np.array(rows, float).reshape(-1, 5)
GT = {s: load_gt(s) for s in man.image_id}
print(len(GT), 'images,', sum(len(v) for v in GT.values()), 'GT boxes,', len(boards), 'boards')

1500 images, 10013 GT boxes, 11 boards


## 1. Engine v2 (template-based graph consequence) + automatic tuning

In [ ]:
def copper_mask(img, copper=COPPER):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, t = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = t > 0
    if copper == 'dark' or (copper == 'auto' and m.mean() > 0.5):
        m = ~m
    return remove_small_objects(m, 30)

S8 = np.ones((3, 3)); KER = {t: np.ones((2*t+1, 2*t+1), np.uint8) for t in range(4)}

def masks(stem):
    return copper_mask(cv2.imread(str(temp_path[stem]))), copper_mask(cv2.imread(str(Y/'images'/f'{stem}.jpg')))

def prep(mt, mx, tol, sig):
    mtd = cv2.dilate(mt.astype(np.uint8), KER[tol]).astype(bool) if tol else mt
    mxd = cv2.dilate(mx.astype(np.uint8), KER[tol]).astype(bool) if tol else mx
    lt, nt = ndi.label(mt, structure=S8); lx, nx = ndi.label(mx, structure=S8)
    big = np.bincount(lt.ravel(), minlength=nt + 1) >= MIN_AREA; big[0] = False
    pair = np.bincount(lt.ravel().astype(np.int64) * (nx + 1) + lx.ravel(), minlength=(nt + 1) * (nx + 1)).reshape(nt + 1, nx + 1)
    pair[:, 0] = 0; pair[0, :] = 0
    return dict(mt=mt, mx=mx, mtd=mtd, mxd=mxd, lt=lt, lx=lx, sig=pair >= sig, big=big)

def query2(a, box, pad, minpx):
    # box-LOCALISED: missing/excess copper (beyond alignment tolerance) inside the box, and the split/merge must be visible near the box
    padl = 3 * pad
    x1, y1, x2, y2 = [int(round(v)) for v in box]
    win = lambda p: (slice(max(0, y1-p), min(640, y2+p+1)), slice(max(0, x1-p), min(640, x2+p+1)))
    w1, w2 = win(pad), win(padl)
    missing = a['mt'][w1] & ~a['mxd'][w1]; excess = a['mx'][w1] & ~a['mtd'][w1]
    nm, ne = int(missing.sum()), int(excess.sum())
    g_open = g_short = False
    if nm >= minpx:
        for n in np.unique(a['lt'][w1][missing]):
            if n == 0 or not a['big'][n]: continue
            pcs = np.unique(a['lx'][w2][a['lt'][w2] == n]); pcs = pcs[pcs > 0]
            if a['sig'][n, pcs].sum() >= 2: g_open = True; break
    if ne >= minpx:
        for j in np.unique(a['lx'][w1][excess]):
            if j == 0: continue
            nets = np.unique(a['lt'][w2][a['lx'][w2] == j]); nets = nets[nets > 0]
            if (a['sig'][nets, j] & a['big'][nets]).sum() >= 2: g_short = True; break
    return g_open, g_short, nm, ne

def verdict(go, gs, nm, ne):          # 0 none, 1 open, 2 short ('both' resolved by larger evidence)
    if go and gs: return 1 if nm >= ne else 2
    return 1 if go else (2 if gs else 0)

def eval_params(cache, params):
    tol, sig, pad, minpx = params; rows = []
    for stem, (mt, mx) in cache.items():
        a = prep(mt, mx, tol, sig)
        for c, *bx in GT[stem]:
            rows.append((int(c), verdict(*query2(a, bx, pad, minpx))))
    r = np.array(rows)
    ao = (r[r[:, 0] == 0, 1] == 1).mean(); as_ = (r[r[:, 0] == 1, 1] == 2).mean(); ff = (r[~np.isin(r[:, 0], CRIT), 1] > 0).mean()
    return ao, as_, ff

In [ ]:
pj = f'{D}/05_results/engine_v2_params.json'
J = json.load(open(pj)) if os.path.exists(pj) else {}
if J.get('grid') == GRID_VERSION and not RETUNE:
    ENG = tuple(J['params'][k] for k in ['tol', 'sig', 'pad', 'minpx'])
    ao, as_, ff = J['heldout']['acc_open'], J['heldout']['acc_short'], J['heldout']['false_flag']
    print('loaded saved engine params (set RETUNE=True to redo):', dict(zip(['tol', 'sig', 'pad', 'minpx'], ENG)))
else:
    rng = np.random.default_rng(0)
    tune_stems = list(rng.choice(man[man.fold.isin([0, 1])].image_id.values, 300, replace=False))
    held_stems = list(rng.choice(man[man.fold.isin([2, 3])].image_id.values, 300, replace=False))
    cache_t = {s: masks(s) for s in tune_stems}; cache_h = {s: masks(s) for s in held_stems}
    grid = list(itertools.product([0, 1], [10], [2, 4, 6], [1, 2, 4]))      # tol, sig, pad, minpx  (sig had no effect before)
    res = []
    for p in grid:
        a_o, a_s, f_f = eval_params(cache_t, p)
        res.append(dict(tol=p[0], sig=p[1], pad=p[2], minpx=p[3], acc_open=a_o, acc_short=a_s, false_flag=f_f, obj=(a_o + a_s) / 2 + 0.5 * (1 - f_f)))
    tr = pd.DataFrame(res).sort_values('obj', ascending=False)
    print(tr.head(6).round(3).to_string(index=False))
    best = tr.iloc[0]; ENG = (int(best.tol), int(best.sig), int(best.pad), int(best.minpx))
    ao, as_, ff = eval_params(cache_h, ENG)
    print(f'\nTUNED {dict(zip(["tol", "sig", "pad", "minpx"], ENG))}')
    json.dump(dict(grid=GRID_VERSION, params=dict(zip(['tol', 'sig', 'pad', 'minpx'], ENG)),
                   heldout=dict(acc_open=ao, acc_short=as_, false_flag=ff)), open(pj, 'w'))
    del cache_t, cache_h
print(f'HELD-OUT boards (folds 2-3):  strict acc open={ao:.3f}  short={as_:.3f}  false-flag on non-critical={ff:.3f}')

loaded saved engine params (set RETUNE=True to redo): {'tol': 1, 'sig': 10, 'pad': 2, 'minpx': 2}
HELD-OUT boards (folds 2-3):  strict acc open=0.644  short=0.737  false-flag on non-critical=0.012


In [ ]:
# full-dataset validation with tuned params (reported, includes tuning boards)
rows = []
for stem in man.image_id:
    g = GT[stem]
    if len(g) == 0: continue
    mt, mx = masks(stem); a = prep(mt, mx, ENG[0], ENG[1])
    for c, *bx in g:
        rows.append(dict(stem=stem, cls=int(c), v=verdict(*query2(a, bx, ENG[2], ENG[3]))))
va = pd.DataFrame(rows); va['verdict'] = va.v.map({0: 'none', 1: 'open', 2: 'short'})
print(pd.crosstab(va.cls.map(dict(enumerate(NAMES))), va.verdict, normalize='index').round(3).reindex(NAMES))
va.to_csv(f'{D}/05_results/engine_v2_validation.csv', index=False)

verdict     none   open  short
cls                           
open       0.162  0.722  0.116
short      0.080  0.161  0.759
mousebite  0.983  0.008  0.009
spur       0.964  0.002  0.034
copper     0.998  0.001  0.001
pin-hole   0.991  0.007  0.002


## 2. Detectors (auto-train missing folds) + out-of-fold predictions

In [ ]:
from ultralytics import YOLO
import torch

def exp_name(cfg, fold, seed):
    return f'det_yolo11n_deeppcb{"+synth1" if cfg == "synth1" else ""}_f{fold}_s{seed}_v2'

def build_split(fold, use_synth, seed):
    restore_tar(f'{D}/01_prepared/prepared_v2.tar', f'{L}/data', marker='.ready_prepared_v2')
    m = man.copy(); m['split'] = np.where(m.fold == fold, 'test', np.where(m.fold == (fold + 1) % 4, 'val', 'train'))
    S = Path(f'{L}/data/split'); shutil.rmtree(S, ignore_errors=True)
    for s in ['train', 'val', 'test']:
        (S/'images'/s).mkdir(parents=True); (S/'labels'/s).mkdir(parents=True)
    for r in m.itertuples():
        shutil.copy2(Y/'images'/f'{r.image_id}.jpg', S/'images'/r.split/f'{r.image_id}.jpg')
        shutil.copy2(Y/'labels'/f'{r.image_id}.txt', S/'labels'/r.split/f'{r.image_id}.txt')
    if use_synth:
        restore_tar(f'{D}/02_synth/synth_v2.tar', f'{L}/data', marker='.ready_synth_v2')
        sm = pd.read_csv(f'{L}/data/synth/synth_meta.csv', dtype={'template': str})
        sm = sm[sm.source_board.isin(set(m[m.split == 'train'].source_board))]
        n = int((m.split == 'train').sum()); k = min(n // 2, sm.op.value_counts().min())
        sm = sm.groupby('op').sample(k, random_state=seed)                      # balanced open/short
        for sid in sm.synth_id:
            shutil.copy2(f'{L}/data/synth/images/{sid}.jpg', S/'images'/'train'/f'{sid}.jpg')
            shutil.copy2(f'{L}/data/synth/labels/{sid}.txt', S/'labels'/'train'/f'{sid}.txt')
    yaml.safe_dump(dict(path=str(S), train='images/train', val='images/val', test='images/test',
                        names={i: n for i, n in enumerate(NAMES)}), open(f'{L}/data/data.yaml', 'w'))

def train_one(cfg, fold, seed):
    use_synth = cfg == 'synth1'; exp = exp_name(cfg, fold, seed)
    RD = f'{D}/04_runs/{exp}'; LOCAL = f'{L}/runs/{exp}'
    if os.path.exists(f'{RD}/DONE'): return
    os.makedirs(f'{RD}/weights', exist_ok=True)
    build_split(fold, use_synth, seed)
    last = {'t': 0}
    def mirror(trainer):
        if time.time() - last['t'] > 120:
            for f in ['weights/last.pt', 'weights/best.pt', 'results.csv', 'args.yaml']:
                p = os.path.join(trainer.save_dir, f)
                if os.path.exists(p): safe_copy(p, f'{RD}/{f}', verify=False)
            last['t'] = time.time()
    src, dst = f'{RD}/weights/last.pt', f'{LOCAL}/weights/last.pt'
    if os.path.exists(src) and not os.path.exists(dst): safe_copy(src, dst, verify=False)
    if os.path.exists(dst):
        mdl = YOLO(dst); mdl.add_callback('on_model_save', mirror); mdl.train(resume=True)
    else:
        mdl = YOLO('yolo11n.pt'); mdl.add_callback('on_model_save', mirror)
        mdl.train(data=f'{L}/data/data.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=30, seed=seed,
                  deterministic=True, workers=min(4, os.cpu_count()), project=f'{L}/runs', name=exp, exist_ok=True, cache='disk')
    for f in ['weights/last.pt', 'weights/best.pt', 'results.csv', 'args.yaml']:
        if os.path.exists(f'{LOCAL}/{f}'): safe_copy(f'{LOCAL}/{f}', f'{RD}/{f}')
    r = YOLO(f'{LOCAL}/weights/best.pt').val(data=f'{L}/data/data.yaml', split='test', imgsz=IMGSZ)
    row = pd.DataFrame([dict(exp_id=exp, use_synth=use_synth, synth_ratio=1.0 if use_synth else 0, fold=f'f{fold}', seed=seed,
                             test_map50=r.box.map50, test_map50_95=r.box.map)])
    led = f'{D}/05_results/master_results_v2.csv'
    row.to_csv(led, mode='a', header=not os.path.exists(led), index=False)
    json.dump(dict(exp_id=exp, seed=seed, fold=fold, use_synth=use_synth, torch=torch.__version__), open(f'{RD}/run_info.json', 'w'))
    open(f'{RD}/DONE', 'w').write('ok')

def ensure_detectors(cfg, seed):
    for fold in range(4):
        if os.path.exists(f'{D}/04_runs/{exp_name(cfg, fold, seed)}/DONE'): continue
        assert TRAIN_MISSING, f'missing detector {exp_name(cfg, fold, seed)} and TRAIN_MISSING=False'
        if time.time() - T0 > MAX_TRAIN_HOURS * 3600:
            print(f'training budget reached at fold {fold}; re-run this cell to continue'); return False
        print('training', exp_name(cfg, fold, seed)); train_one(cfg, fold, seed)
    return True

def get_preds(cfg, seed):
    path = f'{D}/05_results/oof_preds_{cfg}_s{seed}.csv'
    if os.path.exists(path): return pd.read_csv(path, dtype={'stem': str})
    out = []
    for k in range(4):
        mdl = YOLO(f'{D}/04_runs/{exp_name(cfg, k, seed)}/weights/best.pt')
        stems = list(man[man.fold == k].image_id)
        for r in mdl.predict([str(Y/'images'/f'{s}.jpg') for s in stems], conf=0.01, iou=0.6, imgsz=640, max_det=100, stream=True, verbose=False, batch=16):
            for (x1, y1, x2, y2), cf, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()):
                out.append((Path(r.path).stem, k, int(c), float(cf), x1, y1, x2, y2))
    p = pd.DataFrame(out, columns=['stem', 'fold', 'cls', 'conf', 'x1', 'y1', 'x2', 'y2']); p.to_csv(path, index=False); return p

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


## 3. Features, reference-free head, evaluation

In [ ]:
def engine_feats(df):
    out = np.zeros((len(df), 4)); xy = df[['x1', 'y1', 'x2', 'y2']].values
    for stem, sub in df.groupby('stem'):
        mt, mx = masks(stem); a = prep(mt, mx, ENG[0], ENG[1])
        for i in sub.index: out[i] = query2(a, xy[i], ENG[2], ENG[3])
    return out

def make_crops(df):
    X = np.zeros((len(df), CROP, CROP), np.uint8); h = CROP // 2
    cx = np.clip(((df.x1 + df.x2) / 2).values.astype(int), 0, 639); cy = np.clip(((df.y1 + df.y2) / 2).values.astype(int), 0, 639)
    for stem, sub in df.groupby('stem'):
        gp = np.pad(cv2.imread(str(Y/'images'/f'{stem}.jpg'), 0), h, mode='reflect')
        for i in sub.index: X[i] = gp[cy[i]:cy[i] + CROP, cx[i]:cx[i] + CROP]
    return X

import torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
def make_net():
    blk = lambda i, o: [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU()]
    return nn.Sequential(*blk(1, 16), nn.MaxPool2d(2), *blk(16, 32), nn.MaxPool2d(2), *blk(32, 64),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 3))

def train_head(X, y, seed=0):
    torch.manual_seed(seed); net = make_net().to(dev); opt = torch.optim.Adam(net.parameters(), 2e-3)
    cnt = np.bincount(y, minlength=3) + 1; w = torch.tensor((cnt.sum() / cnt / 3).clip(0.2, 5), dtype=torch.float32, device=dev)
    lossf = nn.CrossEntropyLoss(weight=w); Xt = torch.tensor(X[:, None] / 255., dtype=torch.float32); yt = torch.tensor(y).long()
    for ep in range(HEAD_EPOCHS):
        net.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), 256):
            idx = perm[i:i + 256]; xb = Xt[idx].to(dev); yb = yt[idx].to(dev)
            if np.random.rand() < .5: xb = torch.flip(xb, [3])
            if np.random.rand() < .5: xb = torch.flip(xb, [2])
            opt.zero_grad(); lossf(net(xb), yb).backward(); opt.step()
    return net

def head_probs(net, X):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), 512):
            out.append(torch.softmax(net(torch.tensor(X[i:i + 512, None] / 255., dtype=torch.float32).to(dev)), 1).cpu().numpy())
    return np.concatenate(out)

def add_features(df, with_head=True):
    df['crit'] = df.cls.isin(CRIT).astype(int); df['conf_crit'] = df.conf * df.crit
    df['logarea'] = np.log((df.x2 - df.x1).clip(1) * (df.y2 - df.y1).clip(1))
    df['g_agree'] = (((df.cls == 0) & (df.g_open > 0)) | ((df.cls == 1) & (df.g_short > 0))).astype(int)
    df['g_cross'] = (((df.cls == 0) & (df.g_short > 0)) | ((df.cls == 1) & (df.g_open > 0))).astype(int)
    df['lm'] = np.log1p(df.missing); df['le'] = np.log1p(df.excess); df['conf_agree'] = df.conf * df.g_agree
    if with_head:
        df['p_agree'] = np.where(df.cls == 0, df.p_open, np.where(df.cls == 1, df.p_short, 0.0)); df['conf_pagree'] = df.conf * df.p_agree
    return df

F_IMG = ['conf', 'crit', 'conf_crit', 'logarea']
F_GRAPH = F_IMG + ['g_open', 'g_short', 'g_agree', 'g_cross', 'lm', 'le', 'conf_agree']
F_HEAD = F_IMG + ['p_open', 'p_short', 'p_agree', 'conf_pagree']
POL = {'A0 raw conf': None, 'A image-only': F_IMG, 'B +graph (template)': F_GRAPH, 'C +head (reference-free)': F_HEAD}

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

POL = {k: v for k, v in POL.items() if RUN_HEAD or not k.startswith('C')}
METHODS = ['pooled', 'worst-board', 'board-median', 'shift-inflated']

def iou_mat(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    A = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); B = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (A[:, None] + B[None, :] - inter + 1e-9)

def build_gt(df):
    groups = {s: sub.index.values for s, sub in df.groupby('stem')}
    y = np.zeros(len(df), int); rows = []
    for stem in man.image_id:
        g = GT[stem]; g = g[np.isin(g[:, 0], CRIT)]
        if len(g) == 0: continue
        idx = groups.get(stem, np.array([], int))
        I = iou_mat(df.loc[idx, ['x1', 'y1', 'x2', 'y2']].values, g[:, 1:5]) if len(idx) else np.zeros((0, len(g)))
        if len(idx): y[idx] = (I.max(1) >= 0.5).astype(int)
        for j in range(len(g)):
            rows.append(dict(stem=stem, board=board_of[stem], pred_idx=idx[I[:, j] >= 0.5] if len(idx) else np.array([], int)))
    df['y'] = y
    return pd.DataFrame(rows)

gt_scores = lambda sc, sub: np.array([sc[i].max() if len(i) else -1.0 for i in sub.pred_idx])

def tau_for(s, alpha):                               # split-conformal quantile of critical-defect scores
    s = np.sort(s); k = int(np.floor(alpha * (len(s) + 1)))
    return -np.inf if k < 1 else s[k - 1]

def miss_frac(s, tau): return float(((s < tau) | (s == -1)).mean())

def taus_for(s_by, al):
    per = [tau_for(s, al) for s in s_by]; allc = np.concatenate(s_by)
    ratios = []                                      # leave-one-calibration-board-out shift estimate (calibration boards only)
    for j in range(len(s_by)):
        t = tau_for(np.concatenate([s for k, s in enumerate(s_by) if k != j]), al)
        ratios.append(miss_frac(s_by[j], t) / al)
    lam = float(np.clip(np.mean(ratios), 1.0, 3.0))
    return {'pooled': tau_for(allc, al), 'worst-board': min(per), 'board-median': float(np.median(per)),
            'shift-inflated': tau_for(allc, al / lam)}

def lobo(df, gt):
    gtb = {b: gt[gt.board == b] for b in order}; rows = []
    for i, b in enumerate(order):
        calib = [order[(i + k) % 11] for k in (1, 2, 3)]; fit = [x for x in order if x != b and x not in calib]
        m_fit = df.board.isin(fit).values; mte = (df.board == b).values; n_img = int((man.source_board == b).sum())
        for pol, feats in POL.items():
            if feats is None: sc = (df.conf * df.crit).values
            else:
                mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
                mdl.fit(df.loc[m_fit, feats].fillna(0), df.loc[m_fit, 'y']); sc = mdl.predict_proba(df[feats].fillna(0))[:, 1]
            s_by = [gt_scores(sc, gtb[c]) for c in calib]; s_te = gt_scores(sc, gtb[b])
            for al in ALPHAS:
                for meth, tau in taus_for(s_by, al).items():
                    flag = mte & (sc >= tau)
                    rows.append(dict(policy=pol, method=meth, alpha=al, board=b, n_gt=len(s_te),
                                     n_miss=int(((s_te < tau) | (s_te == -1)).sum()), flagged=int(flag.sum()), n_img=n_img, tp=int(df.y[flag].sum())))
    return pd.DataFrame(rows)

def boot_multi(Rall, ref='A image-only'):
    rng = np.random.default_rng(1); idx = rng.integers(0, 11, (N_BOOT, 11)); out = []
    for (cfg_, al, meth), g in Rall.groupby(['cfg', 'alpha', 'method']):
        seeds = sorted(g.seed.unique()); pols = list(g.policy.unique())
        def arrs(p):
            a = g[g.policy == p]
            piv = lambda col: np.stack([a[a.seed == s].set_index('board').loc[order][col].values for s in seeds])
            return [piv(c) for c in ['n_miss', 'n_gt', 'flagged', 'n_img', 'tp']]
        A = {p: arrs(p) for p in pols}
        def stat(p):
            nm, ng, fl, ni, _ = A[p]
            return (np.stack([nm[k][idx].sum(1) / ng[k][idx].sum(1) for k in range(len(seeds))]),
                    np.stack([fl[k][idx].sum(1) / ni[k][idx].sum(1) for k in range(len(seeds))]))
        mref, lref = stat(ref)
        for p in pols:
            m, l = stat(p); nm, ng, fl, ni, tp = A[p]
            dl, dm = l - lref, m - mref; dls, dms = dl.mean(0), dm.mean(0)
            out.append(dict(cfg=cfg_, alpha=al, method=meth, policy=p, n_seeds=len(seeds),
                            miss_over_alpha=nm.sum() / ng.sum() / al, load=fl.sum() / ni.sum(), precision=tp.sum() / max(1, fl.sum()),
                            d_load=dls.mean(), d_load_lo=np.percentile(dls, 2.5), d_load_hi=np.percentile(dls, 97.5),
                            rel_load=dls.mean() / lref.mean(), d_miss=dms.mean(), d_miss_hi=np.percentile(dms, 97.5),
                            seeds_load_better=int((dl.mean(1) < 0).sum()), seeds_ci_excl0=int((np.percentile(dl, 97.5, axis=1) < 0).sum()),
                            per_seed_dload=[round(float(x), 2) for x in dl.mean(1)]))
    return pd.DataFrame(out)

## Run: training queue, evaluation, results

In [ ]:
# ---- training queue (resume-safe). Re-run this cell until it prints "queue complete". ----
queue = [(c, s) for c in CONFIGS for s in SEEDS]
complete = True
for c, s in queue:
    if not ensure_detectors(c, s):
        complete = False; break
done = [(c, s) for c, s in queue if all(os.path.exists(f'{D}/04_runs/{exp_name(c, f, s)}/DONE') for f in range(4))]
print('queue complete' if complete else 'queue NOT complete - re-run this cell', '| ready for evaluation:', done)

training det_yolo11n_deeppcb_f0_s44_v2
Ultralytics 8.4.176 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=det_yolo11n_deeppcb_f0_s44_v2

In [ ]:
def run_one(cfg, seed):
    path = f'{D}/05_results/lobo07_{cfg}_s{seed}.csv'
    if os.path.exists(path): return pd.read_csv(path)
    df = get_preds(cfg, seed).reset_index(drop=True); df['board'] = df.stem.map(board_of)
    df[['g_open', 'g_short', 'missing', 'excess']] = engine_feats(df)
    if RUN_HEAD:
        X = make_crops(df); yt = np.array([verdict(a > 0, b > 0, c, d) for a, b, c, d in df[['g_open', 'g_short', 'missing', 'excess']].values])
        pr = np.zeros((len(df), 3))
        for h in range(4):
            tr, te = (df.fold != h).values, (df.fold == h).values
            pr[te] = head_probs(train_head(X[tr], yt[tr]), X[te])
        df['p_open'], df['p_short'] = pr[:, 1], pr[:, 2]
    df = add_features(df, with_head=RUN_HEAD); gt = build_gt(df)
    R = lobo(df, gt); R.insert(0, 'seed', seed); R.insert(0, 'cfg', cfg)
    R.to_csv(path, index=False)
    print(f'{cfg} s{seed}: {len(gt)} critical GT, {int((gt.pred_idx.map(len) == 0).sum())} unmatched')
    return R

Rall = pd.concat([run_one(c, s) for c, s in done], ignore_index=True) if done else pd.DataFrame()
S = boot_multi(Rall) if len(Rall) else pd.DataFrame()
if len(S): S.to_csv(f'{D}/05_results/replication_summary.csv', index=False)
print('evaluated', sorted(set(zip(Rall.cfg, Rall.seed))) if len(Rall) else 'nothing yet')

real s42: 3448 critical GT, 97 unmatched
real s43: 3448 critical GT, 105 unmatched
real s44: 3448 critical GT, 104 unmatched
synth1 s42: 3448 critical GT, 92 unmatched
synth1 s43: 3448 critical GT, 107 unmatched
synth1 s44: 3448 critical GT, 95 unmatched
evaluated [('real', 42), ('real', 43), ('real', 44), ('synth1', 42), ('synth1', 43), ('synth1', 44)]


In [ ]:
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 40)
assert len(S), 'no finished (config, seed) yet - run the training queue cell first'
show = ['A image-only', 'B +graph (template)'] + (['C +head (reference-free)'] if RUN_HEAD else [])
for cfg_ in CONFIGS:
    for al in ALPHAS:
        g = S[(S.cfg == cfg_) & (S.alpha == al) & (S.policy.isin(show))]
        if not len(g): continue
        print(f"\n===== config={cfg_}  alpha={al}  seeds={int(g.n_seeds.iloc[0])} =====")
        print(g[['method', 'policy', 'miss_over_alpha', 'load', 'precision', 'd_load', 'd_load_lo', 'd_load_hi', 'rel_load', 'd_miss', 'seeds_load_better', 'seeds_ci_excl0', 'per_seed_dload']]
              .round(3).to_string(index=False))
print('\n(d_* = policy minus A image-only; negative = better; CI = 95% bootstrap over the 11 boards, averaged over seeds)')

In [ ]:
# ---- Part 2: which calibration method is closest to 1.0x alpha with moderate load? ----
B = 'B +graph (template)'
tab = (S[S.policy.isin(['A image-only', B])].groupby(['method', 'policy'])
        .agg(miss_over_alpha=('miss_over_alpha', 'mean'), load=('load', 'mean')).unstack('policy').round(3))
print(tab.to_string())
cand = S[(S.policy == B)].groupby('method').agg(m=('miss_over_alpha', 'mean'), load=('load', 'mean'))
ok = cand[(cand.m >= 0.95) & (cand.m <= 1.10)]
if len(ok):
    best = ok.load.idxmin(); print(f"\nRECOMMENDED: {best}  (B miss/alpha = {ok.loc[best, 'm']:.2f}, load = {ok.loc[best, 'load']:.2f} boxes/img)")
else:
    near = cand.assign(gap=(cand.m - 1.0).abs()).sort_values('gap')
    print('\nNo method lands in [0.95, 1.10] for B; closest to 1.0x:'); print(near.round(3).to_string())

               miss_over_alpha                             load                    
policy            A image-only B +graph (template) A image-only B +graph (template)
method                                                                             
board-median             1.599               1.220        2.675               2.452
pooled                   1.082               1.004        4.046               3.178
shift-inflated           0.893               0.925        5.949               4.808
worst-board              0.673               0.669        8.558               7.773

RECOMMENDED: pooled  (B miss/alpha = 1.00, load = 3.18 boxes/img)


In [ ]:
# ---- Part 1 verdict: does the load reduction hold across seeds? ----
holds = {}
print('B vs image-only, rule: better in EVERY seed, combined CI < 0, miss not worse (<= +0.005), per-seed CI < 0 in >= half of seeds\n')
for (cfg_, al, meth), g in S[S.policy == B].groupby(['cfg', 'alpha', 'method']):
    r = g.iloc[0]; n = int(r.n_seeds)
    ok = (n >= 2 and r.seeds_load_better == n and r.d_load_hi < 0 and r.d_miss <= 0.005 and r.seeds_ci_excl0 >= int(np.ceil(n / 2)))
    holds[(cfg_, al, meth)] = bool(ok)
    print(('HOLDS  ' if ok else 'fails  ') + f'{cfg_:6s} alpha={al:<5} {meth:15s} seeds={n} | rel load change {r.rel_load:+.1%} | d_load {r.d_load:+.2f} [{r.d_load_lo:+.2f},{r.d_load_hi:+.2f}] | per-seed {r.per_seed_dload}')
core = [v for (c, a, m), v in holds.items() if m in ('pooled', 'shift-inflated')]
print('\nOVERALL:', 'load reduction holds across seeds for all core settings -> keep the result' if core and all(core)
      else ('holds for some settings only -> report exactly which, and treat the effect as setting-dependent' if any(core)
            else 'does NOT hold across seeds -> drop the claim'))
json.dump({'|'.join(map(str, k)): v for k, v in holds.items()}, open(f'{D}/05_results/replication_verdict.json', 'w'), indent=1)

B vs image-only, rule: better in EVERY seed, combined CI < 0, miss not worse (<= +0.005), per-seed CI < 0 in >= half of seeds

HOLDS  real   alpha=0.05  board-median    seeds=3 | rel load change -14.0% | d_load -0.42 [-0.79,-0.12] | per-seed [-0.47, -0.26, -0.52]
HOLDS  real   alpha=0.05  pooled          seeds=3 | rel load change -26.8% | d_load -1.52 [-2.67,-0.48] | per-seed [-1.79, -0.89, -1.87]
HOLDS  real   alpha=0.05  shift-inflated  seeds=3 | rel load change -16.4% | d_load -1.42 [-3.23,-0.35] | per-seed [-2.54, -0.37, -1.36]
HOLDS  real   alpha=0.05  worst-board     seeds=3 | rel load change -20.0% | d_load -1.96 [-3.72,-0.61] | per-seed [-3.41, -1.0, -1.45]
fails  real   alpha=0.1   board-median    seeds=3 | rel load change +2.4% | d_load +0.05 [-0.14,+0.21] | per-seed [0.06, 0.06, 0.03]
fails  real   alpha=0.1   pooled          seeds=3 | rel load change -10.5% | d_load -0.26 [-0.55,-0.03] | per-seed [-0.26, -0.19, -0.34]
fails  real   alpha=0.1   shift-inflated  seeds=3 | rel 

In [ ]:
# detector comparison across seeds: real vs synthetic (test mAP50-95, fold-level)
led = pd.read_csv(f'{D}/05_results/master_results_v2.csv')
led = led[led.fold.astype(str).str.match(r'^f\d$')].drop_duplicates(['exp_id'], keep='last')
if led.use_synth.nunique() == 2:
    pv = led.pivot_table(index=['fold', 'seed'], columns='use_synth', values='test_map50_95', aggfunc='mean').dropna()
    d = (pv[True] - pv[False]).values
    bs = np.random.default_rng(2).choice(d, (5000, len(d))).mean(1)
    print(pv.round(4)); print(f'\nsynth - real mAP50-95: {d.mean():+.4f}, 95% bootstrap CI [{np.percentile(bs, 2.5):+.4f}, {np.percentile(bs, 97.5):+.4f}] over {len(d)} fold-seeds; synth better in {(d > 0).sum()}/{len(d)}')
else:
    print('need both real and synthetic detectors for the comparison')

use_synth   False   True 
fold seed                
f0   42    0.5998  0.6342
     43    0.6344  0.6158
     44    0.6262  0.6578
f1   42    0.6340  0.6268
     43    0.6204  0.6534
     44    0.6014  0.6118
f2   42    0.6474  0.6835
     43    0.7573  0.7406
     44    0.7454  0.7398
f3   42    0.7348  0.7109
     43    0.7194  0.7180
     44    0.7650  0.7495

synth - real mAP50-95: +0.0047, 95% bootstrap CI [-0.0076, +0.0176] over 12 fold-seeds; synth better in 5/12


## How to read the output
- **Replication verdict** is the decision: `HOLDS` on the core settings (pooled and shift-inflated) means the graph-feature load reduction is consistent across detector seeds; otherwise report it as setting-dependent or drop it.
- **Calibration table:** pick the method with B miss/alpha near 1.0 (the notebook recommends one in [0.95, 1.10]) and note its load. Pooled usually under-covers under board shift; worst-board over-covers.
- Limits that remain: 11 boards, DeepPCB (partly artificial defects, research-only licence), raster-component nets, engine parameters tuned on boards that also appear as test boards, and the seed-42 synthetic detectors possibly trained with unbalanced sampling.
- This is evidence about the method on this dataset. It is not a patentability opinion; prior-art search, VIT IP policy and a registered patent agent come before any publication.